## **Performance**

In [2]:
import torch
from torch import nn
from petorch.utilities import get_model_size_in_bytes

dim = 1024
m = nn.Linear(dim,dim, dtype=torch.float16)
assert (size:=get_model_size_in_bytes(m)) == (dim*dim+dim)*2
size

2099200

In [3]:
import time
import tqdm
def cal_time(n,desc, func, *args, **kwargs) -> int:
    seconds = 0
    for _ in tqdm.tqdm(range(n), desc=desc):
        start = time.time()
        _ = func(*args, **kwargs)
        seconds += time.time() - start
    return seconds

### **BitsAndBytes**

In [13]:
from bitsandbytes.nn import Params4bit
from bitsandbytes import functional as F
block_size = 64
n = 10000

def _create_params(device="cpu")->Params4bit:
    return Params4bit(m.weight.data, blocksize=block_size,compress_statistics=True, quant_type="nf4", quant_storage=torch.uint8).to(
        device=device)

def bnb_quant_dequant(device: str) -> torch.Tensor:
    param = _create_params(device)
    org = F.dequantize_4bit(param, param.quant_state)
    assert org.shape == m.weight.shape
    return org

bnb_t_cpu = cal_time(n//100,"bnb_t_cpu", bnb_quant_dequant,"cpu")
print(bnb_t_cpu)

bnb_t_cuda = cal_time(n,"bnb_t_cuda", bnb_quant_dequant,"cuda")
print(bnb_t_cuda)

# Dequant only
def bnb_dequant(param: Params4bit):
    org = F.dequantize_4bit(param, param.quant_state)
    assert org.shape == m.weight.shape
    return org

bnb_t_cpu_do = cal_time(n//100,"bnb_t_cpu_do", bnb_dequant,_create_params("cpu"))
print(bnb_t_cpu_do)

bnb_t_cuda_do = cal_time(n,"bnb_t_cuda_do", bnb_dequant,_create_params("cuda"))
print(bnb_t_cuda_do)


bnb_t_cpu: 100%|██████████| 100/100 [00:06<00:00, 16.03it/s]


6.194416284561157


bnb_t_cuda: 100%|██████████| 10000/10000 [00:09<00:00, 1097.23it/s]


9.033900260925293


bnb_t_cpu_do: 100%|██████████| 100/100 [00:00<00:00, 210.74it/s]


0.4705235958099365


bnb_t_cuda_do: 100%|██████████| 10000/10000 [00:01<00:00, 7035.46it/s]

1.3895196914672852


In [42]:
from petorch.utilities import get_model_size_in_bytes
param2 = _create_params()
print(param2.dtype)
print(param2[:5])
param2.numel()*param2.element_size()

param3 = param2.to(dtype=torch.float16)
print(param3.dtype)
print(param3[:5])
param3.numel()*param3.element_size()

torch.uint8
tensor([[222],
        [254],
        [ 83],
        [ 51],
        [156]], dtype=torch.uint8)
torch.float16
tensor([[222.],
        [254.],
        [ 83.],
        [ 51.],
        [156.]], dtype=torch.float16)


1048576

In [43]:
cal_time(1000,f"{param2.dtype}-{param2.device}", bnb_dequant,param2)

torch.uint8-cpu: 100%|██████████| 1000/1000 [00:04<00:00, 240.13it/s]


4.133362770080566

In [45]:
param2_cuda = param2.cuda()
cal_time(1000,f"{param2_cuda.dtype}-{param2_cuda.device}", bnb_dequant,param2_cuda)

torch.uint8-cuda:0: 100%|██████████| 1000/1000 [00:00<00:00, 6611.08it/s]


0.14840197563171387

In [51]:
# dtype other than uint8 only available for cuda, not cpu.
param2_bfloat16 = param2.to(dtype=torch.bfloat16, device="cuda")
cal_time(1000,f"{param2_bfloat16.dtype}-{param2_bfloat16.device}", bnb_dequant,param2_bfloat16)

torch.bfloat16-cuda:0: 100%|██████████| 1000/1000 [00:00<00:00, 7478.28it/s]


0.13112521171569824

In [54]:
try:
    param2_bfloat16 = param2.to(dtype=torch.bfloat16, device="cpu")
    cal_time(1000, f"{param2_bfloat16.dtype}-{param2_bfloat16.device}", bnb_dequant, param2_bfloat16)
except Exception as e:
    print("Error: ",e)


torch.bfloat16-cpu:   0%|          | 0/1000 [00:00<?, ?it/s]

Error:  The size of tensor a (32768) must match the size of tensor b (16384) at non-singleton dimension 0


### **TorchAo**

In [14]:
from torchao.dtypes import to_nf4, NF4Tensor
def _create_ao_param(device):
    return to_nf4(m.weight.to(device=device),block_size=block_size,scaler_block_size=256)

def torchao_quant_dequant(device:str):
    params = _create_ao_param(device)
    org =  params.get_original_weight()
    assert org.shape == m.weight.shape
    return org

ao_t_cpu =cal_time(n//100,"ao_t_cpu", torchao_quant_dequant,"cpu")
print(ao_t_cpu)

ao_t_cuda = cal_time(n,"ao_t_cuda",torchao_quant_dequant,"cuda" )
print(ao_t_cuda)

# Dequant only

def ao_dequant(param: NF4Tensor):
    org = param.get_original_weight()
    assert org.shape == m.weight.shape
    return org

ao_t_cpu_do = cal_time(n//100,"ao_t_cpu_do", ao_dequant, _create_ao_param("cpu"))
print(ao_t_cpu_do)

ao_t_cuda_do = cal_time(n,"ao_t_cuda_do", ao_dequant, _create_ao_param("cuda"))
print(ao_t_cuda_do)

ao_t_cpu: 100%|██████████| 100/100 [00:05<00:00, 19.37it/s]


4.977970123291016


ao_t_cuda: 100%|██████████| 10000/10000 [00:47<00:00, 211.46it/s]


46.97204899787903


ao_t_cpu_do: 100%|██████████| 100/100 [00:00<00:00, 300.60it/s]


0.3296546936035156


ao_t_cuda_do: 100%|██████████| 10000/10000 [00:03<00:00, 2906.14it/s]

3.3991308212280273


## **Go deeper**

### **BitsAndBytes**

In [13]:
from bitsandbytes.nn import Params4bit
m_bnb = nn.Linear(dim, dim, dtype=torch.float16)
m_bnb.weight = Params4bit(
    m_bnb.weight.data,
    compress_statistics=True,
    quant_type="nf4",
    quant_storage=torch.uint8
).to(device="cpu")
assert (bnb_size:=get_model_size_in_bytes(m_bnb) )== (dim*dim*0.5)+dim*2
bnb_size

526336

In [14]:
from bitsandbytes import functional as F
weight_org = F.dequantize_4bit(m_bnb.weight, m_bnb.weight.quant_state)
for name, params in m_bnb.named_parameters():
    print(name, params.shape)
print(weight_org.shape)

weight torch.Size([524288, 1])
bias torch.Size([1024])
torch.Size([1024, 1024])


### **TorchAO**

In [15]:
from torchao.dtypes import to_nf4, NF4Tensor
m_ao = nn.Linear(dim, dim, dtype=torch.float16)
m_ao.weight = nn.Parameter(to_nf4(m_ao.weight))
ao_size = get_model_size_in_bytes(m_ao)
# assert (ao_size) == (256 * 256 * 0.5) + 256 * 2
ao_size

542882

In [16]:
n = 0
for attr_name in m_ao.weight.__tensor_flatten__()[0]:
    sub_tensor = getattr(m_ao.weight, attr_name)
    print(attr_name, sub_tensor.shape, sub_tensor.dtype)
    n+= sub_tensor.numel()*sub_tensor.element_size()
n+dim*2

quantized_data torch.Size([524288]) torch.uint8
scaler_mean torch.Size([]) torch.float16
quantization_factor torch.Size([64]) torch.float16
quantized_scalers torch.Size([16384]) torch.int8
nf4 torch.Size([16]) torch.float16


542882

In [17]:
print(type(t:=nn.Parameter(to_nf4(nn.Linear(256, 256, dtype=torch.float16).weight))))
print(issubclass(NF4Tensor, nn.Parameter), isinstance(t,nn.Parameter))

<class 'torchao.dtypes.nf4tensor.NF4Tensor'>
False True


In [18]:
for name, params in m_ao.named_parameters():
    print(params.shape, params.element_size())
ao_org_weight = m_ao.weight.get_original_weight()
print(ao_org_weight.shape, ao_org_weight.device, ao_org_weight.element_size())

torch.Size([1024, 1024]) 2
torch.Size([1024]) 2
torch.Size([1024, 1024]) cpu 2


## **Mesure Accuracies**

In [ ]:
org_weight = nn.Linear(dim,dim).weight.detach()
_bnb_quant_weight = Params4bit(
    org_weight.data,
    compress_statistics=True,
    quant_type="nf4",
    quant_storage=torch.uint8
).to(device="cpu")
bnb_nf4_weight = F.dequantize_4bit( _bnb_quant_weight, _bnb_quant_weight.quant_state).detach()
ao_nf4_weight = to_nf4(org_weight).get_original_weight().detach()
print(org_weight.shape, bnb_nf4_weight.shape, ao_nf4_weight.shape)

In [ ]:
import torch
import numpy as np

def compare_tensors(original: torch.Tensor, deq_a: torch.Tensor, deq_b: torch.Tensor):
    """
    Compare quantized-dequantized tensors against the original.
    Prints multiple error/similarity metrics.
    """

    def metrics(ref, test, name):
        diff = ref - test
        ref_np, test_np, diff_np = ref.cpu().numpy(), test.cpu().numpy(), diff.cpu().numpy()

        print(f"\n=== {name} vs Original ===")
        print(f"L1 norm:            {torch.norm(diff, p=1).item():.6e}")
        print(f"L2 norm:            {torch.norm(diff, p=2).item():.6e}")
        print(f"Mean Squared Error: {torch.mean(diff**2).item():.6e}")
        print(f"Root MSE:           {torch.sqrt(torch.mean(diff**2)).item():.6e}")
        print(f"Mean Abs Error:     {torch.mean(torch.abs(diff)).item():.6e}")
        print(f"Max Abs Error:      {torch.max(torch.abs(diff)).item():.6e}")

        # relative error
        print(f"Relative L2 error:  {(torch.norm(diff,2)/torch.norm(ref,2)).item():.6e}")

        # cosine similarity
        cos = torch.nn.functional.cosine_similarity(ref.flatten(), test.flatten(), dim=0)
        print(f"Cosine Similarity:  {cos.item():.6f}")

        # correlation (Pearson)
        corr = np.corrcoef(ref_np.flatten(), test_np.flatten())[0,1]
        print(f"Pearson Corr:       {corr:.6f}")

        # SNR
        signal_power = torch.mean(ref**2).item()
        noise_power = torch.mean(diff**2).item()
        snr = 10 * np.log10(signal_power / (noise_power + 1e-12))
        print(f"Signal-to-Noise:    {snr:.2f} dB")

    metrics(original.detach(), deq_a, "A (torchao NF4)")
    metrics(original, deq_b, "B (bitsandbytes NF4)")


In [ ]:
compare_tensors(org_weight, ao_nf4_weight, bnb_nf4_weight)